In [1]:
import json
import pandas as pd
import numpy as np


In [2]:
path = "../data/moskva.geojson/moskva.geojson"

with open(path, encoding="utf-8") as f:
    data = json.load(f)


In [3]:
rows = [feat["properties"] for feat in data["features"]]
df = pd.json_normalize(rows)

In [4]:
df["datetime"] = pd.to_datetime(df["datetime"])
df = df[df["severity"] != "Без пострадавших (нет данных)"].copy()
df["fatal"] = (df["severity"] == "С погибшими").astype(int)

In [5]:
df["year"] = df["datetime"].dt.year
train = df[df["year"] <= 2023].copy()
val   = df[df["year"] == 2024].copy()
test  = df[df["year"] == 2025].copy()

print(len(train), len(val), len(test))
print(train["fatal"].mean(), val["fatal"].mean(), test["fatal"].mean())

77181 8859 8352
0.04652699498581257 0.03623433796139519 0.03412356321839081


In [6]:
def add_time_features(d):
    d["hour"] = d["datetime"].dt.hour
    d["weekday"] = d["datetime"].dt.day_of_week
    d["month"] = d["datetime"].dt.month
    return d

for d in (train, val, test):
    add_time_features(d)

In [7]:
cat_cols = ["light", "category", "region", "hour", "weekday", "month"]
MIN_COUNT = 100

for col in cat_cols:
    counts = train[col].value_counts()
    keep = counts[counts >= MIN_COUNT].index
    for d in (train, val, test):
        d[col] = d[col].where(d[col].isin(keep), "other")

In [8]:
X_train_cat = pd.get_dummies(train[cat_cols].astype(str), dtype=int)
X_val_cat   = pd.get_dummies(val[cat_cols].astype(str), dtype=int).reindex(columns=X_train_cat.columns, fill_value=0)
X_test_cat  = pd.get_dummies(test[cat_cols].astype(str), dtype=int).reindex(columns=X_train_cat.columns, fill_value=0)

print(X_train_cat.shape, X_val_cat.shape, X_test_cat.shape)

(77181, 200) (8859, 200) (8352, 200)


In [9]:
list_cols = ["weather", "road_conditions", "nearby", "participant_categories"]

keep_vals = {}
for col in list_cols:
    counts = train[col].explode().value_counts()
    keep_vals[col] = counts[counts >= MIN_COUNT].index.tolist()
    print(col, len(keep_vals[col]))

weather 4
road_conditions 12
nearby 38
participant_categories 6


In [10]:
def multi_hot(d, col, keep):
    exploded = d[col].explode()
    exploded = exploded[exploded.isin(keep)]
    dummies = pd.get_dummies(exploded, prefix=col, dtype=int)
    dummies = dummies.groupby(level=0).max()
    return dummies.reindex(index=d.index, columns=[f"{col}_{v}" for v in keep], fill_value=0)

In [11]:
def list_features(d):
    return pd.concat([multi_hot(d, col, keep_vals[col]) for col in list_cols], axis=1)

X_train_list = list_features(train)
X_val_list   = list_features(val)
X_test_list  = list_features(test)

X_train = pd.concat([X_train_cat, X_train_list], axis=1)
X_val   = pd.concat([X_val_cat,   X_val_list],   axis=1)
X_test  = pd.concat([X_test_cat,  X_test_list],  axis=1)

print(X_train.shape, X_val.shape, X_test.shape)

(77181, 260) (8859, 260) (8352, 260)


In [12]:
num_cols = ["participants_count"]
mean = train[num_cols].mean()
std  = train[num_cols].std()

for X, d in ((X_train, train), (X_val, val), (X_test, test)):
    X[num_cols] = (d[num_cols] - mean) / std

print(X_train.shape, X_train.isna().sum().sum())

(77181, 261) 0


In [13]:
np.savez("../data/features.npz",
         X_train=X_train.to_numpy(dtype=np.float64), y_train=train["fatal"].to_numpy(),
         X_val=X_val.to_numpy(dtype=np.float64),     y_val=val["fatal"].to_numpy(),
         X_test=X_test.to_numpy(dtype=np.float64),   y_test=test["fatal"].to_numpy(),
         columns=np.array(X_train.columns))